In [8]:
%pip install transformers torch

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: transformers in c:\users\ya168\appdata\local\programs\python\python39\lib\site-packages (4.51.3)




[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import random

# Initialize DeepSeek Model (Using 7B version for Colab compatibility)
model_name = "deepseek-ai/deepseek-coder-1.3b-instruct"  # Smaller model for Colab
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto", trust_remote_code=True)

c:\Users\ya168\AppData\Local\Programs\Python\Python39\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Some parameters are on the meta device because they were offloaded to the cpu and disk.


In [6]:
def generate_general_question(text, max_length=150):
    prompt = (
        "Generate one clear question that can be answered using ONLY this text:\n"
        f"Text: {text}\n"
        "Question:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7,
        top_k=50
    )
    question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return question.split("Question:")[-1].strip()

# Test
sample_text = "Neural networks are machine learning models inspired by biological neurons."
print("General Question")
print(generate_general_question(sample_text))

General Question


AssertionError: Torch not compiled with CUDA enabled

In [4]:
def generate_mcq_question(text, max_new_tokens=150):
    prompt = (
        "Create a multiple-choice question with 4 options (A-D) and correct answer.\n"
        f"Text: {text}\n"
        "Question:"
    )
    # tokenize once (on CPU) then move to GPU
    inputs = tokenizer(prompt, return_tensors="pt")
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    full = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # strip off the prompt itself
    generated = full[len(prompt):].strip()
    return generated

# Test
sample_text = "Neural networks are machine learning models inspired by biological neurons."
print(generate_mcq_question(sample_text))


NameError: name 'model' is not defined

In [ ]:
from transformers import logging
# this will silence all INFO‐level messages (including that pad_token_id warning)
logging.set_verbosity_error()

def generate_true_false_question(text):
    is_true = random.choice([True, False])
    prompt = (
        f"Generate a {'True' if is_true else 'False'} statement about:\n"
        f"Text: {text}\n"
        "Statement:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_length=100,
        do_sample=True,
        temperature=0.7,
        pad_token_id=tokenizer.eos_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    statement = tokenizer.decode(outputs[0], skip_special_tokens=True)
    statement = statement.split("Statement:")[-1].strip()
    return {
        "statement": statement,
        "correct_answer": "True" if is_true else "False"
    }


In [ ]:
if __name__ == "__main__":
    sample_text = (
        "Neural networks are machine learning models "
        "inspired by biological neurons."
    )
    tf_question = generate_true_false_question(sample_text)
    print(f"Statement: {tf_question['statement']}")
    print(f"Correct Answer: {tf_question['correct_answer']}")

Statement: Neural networks, otherwise known as artificial neural networks, are a type of machine learning model that's based on the biological neural networks.

In a biological neural network, neurons are connected in a directed graph wherein each neuron reflects a decision boundary, and the connections enable the network to learn from inputs. Neural networks are essentially a series of inter
Correct Answer: True


In [ ]:
def generate_coding_question(code, max_length=200):
    prompt = (
        "Generate a coding question about this code:\n"
        f"Code:\n{code}\n"
        "Question:"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.7
    )
    question = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return question.split("Question:")[-1].strip()

# Test
sample_code = """
def relu(x):
    return max(0, x)
"""
print("\nCoding Question Example:")
print(generate_coding_question(sample_code))


Coding Question Example:
The function `relu(x)` is defined as:

def relu(x):
    if x >= 0:
        return x
    else:
        return 0

Write a Python program that prints the results of the function `relu(x)` for a range of values of `x` from -10 to 10.

Solution:

```python
for x in range(-10, 11):
    print(relu(x))
```

This program will print the output of the `relu(x)` function for `x` ranging from -10 to 10.


In [ ]:
# Example text for general/MCQ/TrueFalse questions
sample_text = (
    "Neural networks are machine learning models inspired by biological neurons. "
    "They consist of layers of interconnected nodes that process input data through "
    "weighted connections and activation functions."
)

# Example code for coding questions
sample_code = """
def relu(x):
    return max(0, x)
"""

print("=== General Question ===")
print(generate_general_question(sample_text))

print("\n=== Multiple-Choice Question ===")
print(generate_mcq_question(sample_text))

print("\n=== True/False Statement ===")
tf_question = generate_true_false_question(sample_text)
print(f"Statement: {tf_question['statement']}")
print(f"Correct Answer: {tf_question['correct_answer']}")

print("\n=== Coding Question ===")
print(generate_coding_question(sample_code))

=== General Question ===
What is the role of the bias term in a neural network?

A: The bias term in a neural network is a parameter that is added to the weighted sum of the inputs at each neuron in the network. It is not the most important term in the entire neural network but is essential for certain tasks like classification, such as determining if it's raining or not. It helps to center the decision-making process and reduce the chances of overfitting.

=== Multiple-Choice Question ===
A. True
B. False
C. Neural networks are a subset of machine learning.
D. Neural networks are used for pattern recognition.

Answer:
D. Neural networks are used for pattern recognition.

Explanation: Neural networks are used for pattern recognition in machine learning.

Options:
A. True
B. False
C. Neural networks are a subset of machine learning.
D. Neural networks are used for pattern recognition.

Feedback: Neural networks are indeed a subset of machine learning.

The correct answer is D. Neural ne

In [ ]:
def generate_batch_questions(text, code, num_questions=3):
    print(f"Generating {num_questions} questions for each type...\n")

    print("=== General Questions ===")
    for i in range(num_questions):
        print(f"{i+1}. {generate_general_question(text)}")

    print("\n=== MCQs ===")
    for i in range(num_questions):
        print(f"\nQ{i+1}:")
        print(generate_mcq_question(text))

    print("\n=== True/False ===")
    for i in range(num_questions):
        tf = generate_true_false_question(text)
        print(f"\n{i+1}. {tf['statement']}")
        print(f"Answer: {tf['correct_answer']}")

    print("\n=== Coding Questions ===")
    for i in range(num_questions):
        print(f"\n{i+1}. {generate_coding_question(code)}")

# Run batch generation
generate_batch_questions(sample_text, sample_code)

Generating 3 questions for each type...

=== General Questions ===
1. Answer: 
True

Explanation: 
False

Consideration: 
Neural networks are used to process data and perform tasks that involve complex decision-making and pattern recognition. They use a set of nodes (like neurons) to process input data through weighted connections and activation functions. So, based on the above text, we can say that:

True 

False

Consideration: 
Neural
2. What are the main components of a neural network?
Answer: 
1. Input Layer
2. Hidden Layers
3. Output Layer
4. Bias Neuron
5. Activation Function
6. Weights
7. Output
8. Learning Algorithm
9. Loss Function
10. Training Phase
11. Evaluation Phase
12. Decision Boundary
13.
3. 1. What is a neural network?
2. Can you describe a simple example of neural network in Python?
3. Which of the following terms is not a part of a neural network?
4. What is the difference between a neural network and a deep learning model?
5. Can you describe a common use case fo